<a href="https://colab.research.google.com/github/Athul-11/apexplanet-task-1/blob/main/AIML_Lab_Exam_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIML Lab Exam — Google Colab Template

**How to use**
1. Open this notebook in Google Colab.
2. Run the import and upload cells.
3. Inspect the dataset and replace `TARGET_COLUMN` with the column your question asks you to predict.
4. Choose **Classification** or **Regression** based on the problem statement.
5. Run only the matching workflow.
6. Do not choose a model based only on training accuracy. Use the test metrics and inspect class imbalance.

**Important:** If your instructor specifies a particular algorithm, use that algorithm. Keep a copy of the original dataset unchanged.


## 1. Imports and upload the dataset

In [ ]:
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("First 5 rows:")
display(df.head())
print("Shape:", df.shape)
print("\\nData types and non-null counts:")
df.info()
print("\\nMissing values per column:")
print(df.isnull().sum())
print("\\nDuplicate rows:", df.duplicated().sum())

Saving cleaned_titanic_dataset.csv to cleaned_titanic_dataset (1).csv
First 5 rows:


,survived,pclass,sex,age,sibsp,parch,fare,who,adult_male,embark_town,...,embarked_S,class_First,class_Second,class_Third,fare_minmax,age_standard,family_size,is_alone,fare_per_person,record_date
0,0,3,male,22.0,1,0,7.2500,man,True,Southampton,...,True,False,False,True,0.098747,-0.551060,2,0,3.62500,2026-01-01
1,1,1,female,38.0,1,0,71.2833,woman,False,Cherbourg,...,False,True,False,False,0.970901,0.611945,2,0,35.64165,2026-01-01
2,1,3,female,26.0,0,0,7.9250,woman,False,Southampton,...,True,False,False,True,0.107941,-0.260308,1,1,7.92500,2026-01-01
3,1,1,female,35.0,1,0,53.1000,woman,False,Southampton,...,True,True,False,False,0.723239,0.393881,2,0,26.55000,2026-01-01
4,0,3,male,35.0,0,0,8.0500,man,True,Southampton,...,True,False,False,True,0.109644,0.393881,1,1,8.05000,2026-01-01


Shape: (775, 25)
\nData types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 775 entries, 0 to 774
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   survived         775 non-null    int64  
 1   pclass           775 non-null    int64  
 2   sex              775 non-null    object 
 3   age              775 non-null    float64
 4   sibsp            775 non-null    int64  
 5   parch            775 non-null    int64  
 6   fare             775 non-null    float64
 7   who              775 non-null    object 
 8   adult_male       775 non-null    bool   
 9   embark_town      775 non-null    object 
 10  alive            775 non-null    object 
 11  alone            775 non-null    bool   
 12  sex_encoded      775 non-null    int64  
 13  embarked_C       775 non-null    bool   
 14  embarked_Q       775 non-null    bool   
 15  embarked_S       775 non-null    bool   
 16  class_First

## 2. Inspect and select the target column
Change `TARGET_COLUMN` to the exact target column name. If the task is only preprocessing, use the preprocessing-only section near the end.

In [ ]:
print("Column names:", df.columns.tolist())
print(df.describe(include="all").T)

TARGET_COLUMN = "survived"  # CHANGE THIS

# Remove exact duplicate rows only if appropriate for the dataset
df = df.drop_duplicates().copy()

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Remove rows where the target is missing
valid = y.notna()
X = X.loc[valid].copy()
y = y.loc[valid].copy()

# Remove identifier columns ONLY if they are just IDs and not useful predictors
# X = X.drop(columns=["ID"])

print("Feature columns:", X.columns.tolist())
print("Target:", TARGET_COLUMN)
print("Target values/counts:")
print(y.value_counts(dropna=False).head(20))

Column names: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'who', 'adult_male', 'embark_town', 'alive', 'alone', 'sex_encoded', 'embarked_C', 'embarked_Q', 'embarked_S', 'class_First', 'class_Second', 'class_Third', 'fare_minmax', 'age_standard', 'family_size', 'is_alone', 'fare_per_person', 'record_date']
                 count unique          top freq       mean        std  \
survived         769.0    NaN          NaN  NaN   0.408322   0.491843   
pclass           769.0    NaN          NaN  NaN   2.256177   0.849734   
sex                769      2         male  483        NaN        NaN   
age              769.0    NaN          NaN  NaN  29.567516  13.814097   
sibsp            769.0    NaN          NaN  NaN   0.527958   0.993238   
parch            769.0    NaN          NaN  NaN   0.422627   0.842898   
fare             769.0    NaN          NaN  NaN   26.22778  22.583216   
who                769      3          man  443        NaN        NaN   
adult_male       

## 3. Shared preprocessing and train/test split
This preprocessing handles missing numeric values, missing categorical values, categorical encoding, and numeric scaling. The pipeline fits preprocessing only on training data, helping avoid data leakage.

**Choose one workflow below:** Classification (categories) or Regression (continuous number).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

Numeric columns: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_encoded', 'fare_minmax', 'age_standard', 'family_size', 'is_alone', 'fare_per_person']
Categorical columns: ['sex', 'who', 'adult_male', 'embark_town', 'alive', 'alone', 'embarked_C', 'embarked_Q', 'embarked_S', 'class_First', 'class_Second', 'class_Third', 'record_date']


## 4A. Classification workflow
Use for categories such as pass/fail, yes/no, spam/not spam, or species classes. If a class has only one sample, `stratify=y` may fail; inspect class counts and adjust the split if needed.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

results = []
for name, model in models.items():
    clf = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    results.append({"Model": name, "Test Accuracy": accuracy_score(y_test, pred)})

results_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)
display(results_df)

best_name = results_df.iloc[0]["Model"]
print("Highest test accuracy in this comparison:", best_name)

# Refit selected model to show detailed evaluation
best_model = models[best_name]
best_clf = Pipeline([
    ("preprocessing", preprocessor),
    ("model", best_model)
])
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\\n", confusion_matrix(y_test, y_pred))
print("Classification report:\\n", classification_report(y_test, y_pred, zero_division=0))

,Model,Test Accuracy
0,Logistic Regression,1.0
1,SVM,1.0
2,Decision Tree,1.0
3,Random Forest,1.0
4,Neural Network,1.0


Highest test accuracy in this comparison: Logistic Regression
Test accuracy: 1.0
Confusion matrix:\n [[91  0]
 [ 0 63]]
Classification report:\n               precision    recall  f1-score   support

           0       1.00      1.00      1.00        91
           1       1.00      1.00      1.00        63

    accuracy                           1.00       154
   macro avg       1.00      1.00      1.00       154
weighted avg       1.00      1.00      1.00       154



## 4B. Regression workflow
Use this instead of Classification for a continuous numeric target such as house price, sales, or temperature. Do not run the classification and regression workflows together for the same task.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models_reg = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(),
    "Neural Network": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

reg_results = []
for name, model in models_reg.items():
    reg = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    reg.fit(X_train, y_train)
    pred = reg.predict(X_test)
    reg_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

reg_results_df = pd.DataFrame(reg_results).sort_values("RMSE", ascending=True)
display(reg_results_df)
print("Compare lower MAE/RMSE and higher R2; inspect whether the model generalizes.")

,Model,MAE,RMSE,R2
1,Decision Tree,0.000000e+00,0.000000e+00,1.000000
2,Random Forest,0.000000e+00,0.000000e+00,1.000000
0,Linear Regression,2.634977e-16,3.551914e-16,1.000000
4,Neural Network,3.793619e-02,5.478117e-02,0.987523
3,SVR,7.514838e-02,8.572076e-02,0.969448


Compare lower MAE/RMSE and higher R2; inspect whether the model generalizes.


## 5. Preprocessing-only task (optional)
Run this section only if the question asks you to clean/prepare the dataset without training a predictive model. Change `df` only if you want to preserve the original; this example makes a copy.

In [ ]:
clean_df = df.copy()

num_cols_clean = clean_df.select_dtypes(include=np.number).columns
cat_cols_clean = clean_df.select_dtypes(exclude=np.number).columns

for col in num_cols_clean:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in cat_cols_clean:
    if clean_df[col].notna().any():
        clean_df[col] = clean_df[col].fillna(clean_df[col].mode()[0])

print("Missing values after basic cleaning:")
print(clean_df.isnull().sum())
print("Cleaned shape:", clean_df.shape)
display(clean_df.head())


Missing values after basic cleaning:
survived           0
pclass             0
sex                0
age                0
sibsp              0
parch              0
fare               0
who                0
adult_male         0
embark_town        0
alive              0
alone              0
sex_encoded        0
embarked_C         0
embarked_Q         0
embarked_S         0
class_First        0
class_Second       0
class_Third        0
fare_minmax        0
age_standard       0
family_size        0
is_alone           0
fare_per_person    0
record_date        0
dtype: int64
Cleaned shape: (769, 25)


,survived,pclass,sex,age,sibsp,parch,fare,who,adult_male,embark_town,...,embarked_S,class_First,class_Second,class_Third,fare_minmax,age_standard,family_size,is_alone,fare_per_person,record_date
0,0,3,male,22.0,1,0,7.2500,man,True,Southampton,...,True,False,False,True,0.098747,-0.551060,2,0,3.62500,2026-01-01
1,1,1,female,38.0,1,0,71.2833,woman,False,Cherbourg,...,False,True,False,False,0.970901,0.611945,2,0,35.64165,2026-01-01
2,1,3,female,26.0,0,0,7.9250,woman,False,Southampton,...,True,False,False,True,0.107941,-0.260308,1,1,7.92500,2026-01-01
3,1,1,female,35.0,1,0,53.1000,woman,False,Southampton,...,True,True,False,False,0.723239,0.393881,2,0,26.55000,2026-01-01
4,0,3,male,35.0,0,0,8.0500,man,True,Southampton,...,True,False,False,True,0.109644,0.393881,1,1,8.05000,2026-01-01


## 6. Output interpretation — write in your record
- **Classification:** “The dataset was inspected and preprocessed. The data was split into training and testing sets. The model was trained and evaluated on the test set. The test accuracy was ___%. The confusion matrix and classification report were used to inspect its performance.”
- **Regression:** “The dataset was preprocessed and split into training and testing sets. The model was trained and evaluated using MAE ___, RMSE ___, and R² ___.”
- Report your actual output values. A high test accuracy alone does not guarantee a good model, especially with imbalanced classes.
